In [ ]:
import os
import pickle
import re
from time import sleep
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions, 
    EasyOcrOptions, 
    TableStructureOptions
)
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from langchain_core.documents import Document

# 1. 정규식 패턴 사전 컴파일
NEWLINE_PATTERN = re.compile(r'\r\n\d+')

def normalize_newlines(text: str) -> str:
    return NEWLINE_PATTERN.sub('\n', text)

# 2. 파이프라인 옵션 강화
def get_enhanced_pipeline_options():
    pipeline_options = PdfPipelineOptions(
        do_ocr=True,
        do_table_structure=True, # 테이블 구조 분석 활성화
    )
    
    # 테이블 구조 인식 세부 설정
    pipeline_options.table_structure_options.do_cell_matching = True  # 셀 매칭 강화
    
    # OCR 설정 (한국어/영어)
    pipeline_options.ocr_options = EasyOcrOptions(lang=["en", "ko"])
    
    return pipeline_options

def parsing_pdf_by_page_with_docling(path: str, lv1_cat: str, lv2_cat: str):
    path = path.replace("\\", "/")
    filename = os.path.basename(path)
    base_filename = filename.replace(".pdf", "")

    first_sentence = f"This page explains {base_filename} that belongs to {lv1_cat} and {lv2_cat} categories.\n"

    # 컨버터 설정
    converter = DocumentConverter(
        allowed_formats=[InputFormat.PDF],
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=get_enhanced_pipeline_options(),
                backend=PyPdfiumDocumentBackend
            ),
        }
    )

    # 문서 전체 변환 (Docling은 내부적으로 병렬 처리를 지원하므로 한 번에 변환하는 것이 효율적임)
    conv_res  = converter.convert(path)
    docling_document = conv_res .document
    docs = []
    

    # 3. pdfplumber 없이 docling 결과에서 직접 페이지 순회
    # docling_document.pages는 dict 형태이므로 페이지 번호로 접근 가능
    num_pages = len(docling_document.pages)

    for page_no in tqdm(range(1, num_pages + 1), desc=f"Parsing {filename}"):
        # 특정 페이지의 마크다운 추출 (테이블이 마크다운 표 형식으로 변환됨)
        page_md = docling_document.export_to_markdown(page_no=page_no)
        
        # 후처리
        page_md = page_md.replace("", "")
        page_md = normalize_newlines(page_md)
        content = first_sentence + page_md
        
        # LangChain 문서 객체 생성
        lang_doc = Document(
            page_content=content,
            metadata={
                'filename': filename,
                'lv1_cat': lv1_cat,
                'lv2_cat': lv2_cat,
                'page': str(page_no - 1), # 0-indexed 유지용
            }
        )
        docs.append(lang_doc)
        sleep(0.01) # 필요 시 유지

    
    # 저장 로직
    parsed_foldername = f"{lv1_cat}_{lv2_cat}"
    save_path = Path(f"./docs/{parsed_foldername}_r01")
    save_path.mkdir(parents=True, exist_ok=True)
        
    with open(save_path / f"{base_filename}.pkl", 'ab') as file:
        pickle.dump(docs, file)

    return docs

In [16]:
import time
start_time = time.time()
filepath = f"./file/kubota_sample.pdf"
lv1_cat, lv2_cat = "CE", "KUBOTA"


In [ ]:
result = parsing_pdf_by_page_with_docling(path=filepath, lv1_cat=lv1_cat, lv2_cat=lv2_cat)
end_time = time.time() - start_time
print(f"Document converted and tables exported in {end_time:.2f} seconds.")

In [11]:
with open('./docs/CE_KUBOTA_r01/kubota_sample.pkl', 'rb') as file:
    # Load the pickled object from the file
    loaded_object = pickle.load(file)

In [14]:
from IPython.display import Markdown
Markdown(loaded_object[5].page_content)

This page explains kubota_sample that belongs to CE and KUBOTA categories.
## SPECIFICATIONS

| Model                                                    | Model                                                    | Model                                                    | Model                                                    | Model                                                    | KX015-4                               |
|----------------------------------------------------------|----------------------------------------------------------|----------------------------------------------------------|----------------------------------------------------------|----------------------------------------------------------|---------------------------------------|
| Machine weight *1  (cabin / canopy)                      | Machine weight *1  (cabin / canopy)                      | Machine weight *1  (cabin / canopy)                      | Machine weight *1  (cabin / canopy)                      | kg                                                       | 1470 / 1420                           |
| Operating weight *2  (cabin / canopy)                    | Operating weight *2  (cabin / canopy)                    | Operating weight *2  (cabin / canopy)                    | Operating weight *2  (cabin / canopy)                    | kg                                                       | 1545 / 1495                           |
|                                                          | Model                                                    | Model                                                    | Model                                                    | Model                                                    | D782-E3-BH                            |
|                                                          | Type                                                     | Type                                                     | Type                                                     | Type                                                     | Water- r- cooled,diesel engine,E-TVCS |
|                                                          | Output ISO14396                                          | Output ISO14396                                          | Output ISO14396                                          | PS (kW)/rpm                                              | 13.3 (9.8) / 2300                     |
| Engine                                                   | Output ISO9249 NET                                       | Output ISO9249 NET                                       | Output ISO9249 NET                                       | PS (kW)/rpm                                              | 13.1 (9.6) / 2300                     |
|                                                          | Number of cylinders                                      | Number of cylinders                                      | Number of cylinders                                      | Number of cylinders                                      | 3                                     |
|                                                          | Bore × Stroke                                            | Bore × Stroke                                            | Bore × Stroke                                            | mm                                                       | 67 × 73.6                             |
|                                                          | Displacement                                             | Displacement                                             | Displacement                                             | cc                                                       | 778                                   |
|                                                          | Overall width                                            | Overall width                                            | Overall width                                            | mm                                                       | 990                                   |
|                                                          | Overall height (cabin / canopy)                          | Overall height (cabin / canopy)                          | Overall height (cabin / canopy)                          | mm                                                       | 2350 / 2330                           |
|                                                          | Overall length                                           | Overall length                                           | Overall length                                           | mm                                                       | 3710                                  |
|                                                          | Ground clearance                                         | Ground clearance                                         | Ground clearance                                         | mm                                                       | 160                                   |
| Dimensions                                               | Dozer size (width × height)                              | Dozer size (width × height)                              | Dozer size (width × height)                              | mm                                                       | 990 × 230                             |
|                                                          | Rubber shoe width                                        | Rubber shoe width                                        | Rubber shoe width                                        | mm                                                       | 230                                   |
|                                                          | Minimum front swivel radius                              | Minimum front swivel radius                              | Minimum front swivel radius                              | mm                                                       | 1490                                  |
|                                                          | Boom swing angle (left /right) deg                       | Boom swing angle (left /right) deg                       | Boom swing angle (left /right) deg                       | Boom swing angle (left /right) deg                       | 75 / 60                               |
| Hydraulic                                                | P1, P2                                                   |                                                          |                                                          |                                                          | Variable displacement pump            |
| Hydraulic                                                |                                                          | Flow rate /min                                           | Flow rate /min                                           | Flow rate /min                                           | 16.6 × 2                              |
| Hydraulic                                                |                                                          | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | 20.6 (210)                            |
| Hydraulic                                                | P3                                                       |                                                          |                                                          |                                                          | Gear pump                             |
| Hydraulic                                                |                                                          | Flow rate /min                                           | Flow rate /min                                           | Flow rate /min                                           | 10.4                                  |
| System                                                   |                                                          | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | 20.1 (205)                            |
| Hydraulic                                                | Auxiliary                                                | /min Max. flow rate                                      | /min Max. flow rate                                      | /min Max. flow rate                                      | 27.0                                  |
| Hydraulic                                                | (AUX)                                                    | MPa (kgf/cm 2 ) Max. hydr. pressure                      | MPa (kgf/cm 2 ) Max. hydr. pressure                      | MPa (kgf/cm 2 ) Max. hydr. pressure                      | 20.6 (210)                            |
| Hydraulic                                                | Max. digging                                             |                                                          | arm                                                      | kN (kgf)                                                 | 7.3 (740)                             |
| Hydraulic                                                | force                                                    |                                                          | bucket                                                   | kN (kgf)                                                 | 12.7 (1300)                           |
| Hydraulic                                                | Hydraulic reservoir (full)                               | Hydraulic reservoir (full)                               | Hydraulic reservoir (full)                               | Hydraulic reservoir (full)                               | 28                                    |
| Max. travelling speed  km/h                              | Max. travelling speed  km/h                              | Max. travelling speed  km/h                              | Max. travelling speed  km/h                              | Max. travelling speed  km/h                              | 2.1                                   |
| Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | 25.5 (0.26) / 24.5 (0.25)             |
| Swivelling speed rpm                                     | Swivelling speed rpm                                     | Swivelling speed rpm                                     | Swivelling speed rpm                                     | Swivelling speed rpm                                     | 9.1                                   |
| Fuel tank capacity                                       | Fuel tank capacity                                       | Fuel tank capacity                                       | Fuel tank capacity                                       | Fuel tank capacity                                       | 21                                    |
| LpA dB (A)                                               | LpA dB (A)                                               | LpA dB (A)                                               | LpA dB (A)                                               | LpA dB (A)                                               | 78                                    |
| Noise level                                              | LwA (2000/14/EC)                                         |                                                          |                                                          | dB (A)                                                   | 93                                    |
|                                                          | Hand arm system (ISO5349-2:2001)                         |                                                          | Digging                                                  | m/s2 RMS                                                 | <2.5                                  |
|                                                          |                                                          |                                                          | Levelling                                                | m/s2 RMS                                                 | <2.5                                  |
|                                                          |                                                          |                                                          | Driving                                                  | m/s2 RMS                                                 | <2.5                                  |
| Vibration* KX0154  n*3 Whole  Vibration KX015-4 canopy                                                          |                                                          |                                                          | Idling                                                   | m/s2 RMS                                                 | <2.5                                  |
|                                                          | Whole body anopy (ISO2631-1:1997)                                                          |                                                          | Digging                                                  | m/s2 RMS m/s2 RMS oint radiu ing m/s2 RMS Lifting point radius (2m)                                                          | <0.5 <0.5                             |
|                                                          | Lift Point Height                                        | Levelling Lifti                                                          | Driving Ove ving m Over-front                                                          | m/s2 RMS t                                                          | <0.5                                  |
|                                                          |                                                          | Idling Blade Down                                                          | Idling Down                                                          | m/s2 RMS Blade UP m/s RMS Blade UP                                                          | <0.5 de <0 Over-side Blade Down                                       |

*1 With 32.5 kg Kubota original bucket, full tanks, rubber shoe. 
 (035)
 (030)
m ubber shoe
.4 (0.35)

*2 Machine weight with 75 kg operator.
.0 (0.30
.5m al bucket, fu
operator.
.0 (0.30) 2.5 kg 
ne wei
.5m

2.7 (0.27)

–

*3 These values are mesured under specific conditions at maximum engine speed and can deviate, 
 (037)
 (025)
–
 (046)
m depending on the operating status.
 (0 itions at ma
.6 (0.37) ngine speed 
.5 (0.25) – d under spe
g status
.5 (0.46) values 
.0m

0.5m

3.4 (0.35)

3.3 (0.33)
Y

2.3 (0.23)

2.2 (0.22)

2.6 (0.27)

–

Blade UP

–

–

1.7 (0.17)

–

5.4 (0.55)

## 5.3 (0.54)
APAC 0m
TIN 3.
.3 (0.54)
m
LIFTING CAPACITY

Lifting point radius (max.)

Over-front

Over-side

–

–

1.1 (0.12)

–

| KX015-4 cabin Cabin, Rubber v KX015-4 cabin Cabin, Rubber version                   |                           |                           |                           |                             |                             | kN (ton)                    |
|-------------------|---------------------------|---------------------------|---------------------------|-----------------------------|-----------------------------|-----------------------------|
|                   | Lifting point radius (2m) | Lifting point radius (2m) | Lifting point radius (2m) | Lifting point radius (max.) | Lifting point radius (max.) | Lifting point radius (max.) |
| Lift Point Height | Over-front                | Over-front                | Over-side                 | Over-front                  | Over-front                  | Over-side Pit                             |
|                   | Blade Down                | Blade UP                  |                           | Blade Down                  | Blade UP                    | Over Lift Point                             |
| 1.5m              | 3.0 (0.30)                | 3.4 (0.35)                | 2.5 (0.26)                | –                           | –                           | –                           |
| 1.0m              | 4.5 (0.46)                | 3.5 (0.36)                | 2.3 (0.24)                | –                           | –                           | –                           |
| 0.5m              | 5.4 (0.55)                | 3.3 (0.34)                | 2.1 (0.22)                | 2.6 (0.27)                  | 1.6 (0.17)                  | 1.1 (0.11) oint Heigh 1.1 (0.11) Lift Point Height                             |
| 0m                | 5.3 (0.54)                | 3.2 (0.32)                | 2.0 (0.21)                | –                           | –                           | –                           |

*The lifting capacities are based on ISO 10567 and do not exceed 75% of the static tilt load of the machine or 87%

of the hydraulic lifting capacities of the machine.

**The excavator bucket, hook, sling and other lifting accessories are not included on this table.

<!-- image -->

* Working ranges are with Kubota original bucket, without quick coupler.

* Specifications are subject to change without notice for purpose of improvement.

- ★ All images shown are for brochure purposes only. When operating the excavator, wear clothing and equipment in accordance to local legal and safety regulations.

## KUBOTA (U.K.) LTD

Dormer Road,Thame, Oxfordshire, OX9 3UN, U.K.

Phone : 01844-268140

F a x : 01844-216685

<!-- image -->

3360

2250

## WORKING RANGE

3730

3790

<!-- image -->

1490

<!-- image -->

450

510

450

510

*With cabin, rubber shoe and standard arm kN (ton)

2290

1810

990

1490

1090

1450

1090

3710

990

K


240

230

1070


1070
KX15-4\_2010/12


990

990

240

230

2350

2350

KX15-4\_2010/12

KX15-4\_2010/12

# 테이블만 별도 추출

In [ ]:
import os
import pickle
import pandas as pd
from docling.document_converter import DocumentConverter
from time import time
from tqdm.auto import tqdm
from langchain_core.documents import Document
from collections import defaultdict

file_path = filepath
lv1_cat, lv2_cat = "CE", "KUBOTA"

path = file_path.replace("\\", "/")
filename = path.split("/")[-1]


start_time = time()
doc_converter = DocumentConverter()
conv_res = doc_converter.convert(file_path)
tables = defaultdict(list)
# Export table by
for table_ix, table in tqdm(enumerate(conv_res.document.tables)):
    page_num = table.prov[0].page_no if table.prov else "Unknown"
    table_df: pd.DataFrame = table.export_to_dataframe()
    extracted_table = table_df.to_markdown()
    if str(page_num) not in list(tables.keys()):
        tables[str(page_num)] = [extracted_table]
    else:
        tables[str(page_num)].append(extracted_table)
    
end_time = time() - start_time
print(f"Document converted and tables exported in {end_time:.2f} seconds.")

2026-01-18 13:33:59,783 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]


2026-01-18 13:33:59,834 - INFO - Going to convert document batch...
2026-01-18 13:33:59,836 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2026-01-18 13:33:59,839 - INFO - rapidocr cannot be used because onnxruntime is not installed.
2026-01-18 13:33:59,842 - INFO - Accelerator device: 'cpu'
2026-01-18 13:34:02,019 - INFO - Auto OCR model selected easyocr.
2026-01-18 13:34:02,019 - INFO - Accelerator device: 'cpu'
2026-01-18 13:34:03,007 - INFO - Accelerator device: 'cpu'
2026-01-18 13:34:03,455 - INFO - Processing document kubota_sample.pdf
d:\auto_vectordb\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
d:\auto_vectordb\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pi

Document converted and tables exported in 158.91 seconds.


In [24]:
tables = dict(tables)
tables

{'6': ['|    | Model.Machine weight *1 (cabin / canopy)                 | Model.Machine weight *1 (cabin / canopy)                 | Model.Machine weight *1 (cabin / canopy)                 | Model.Machine weight *1 (cabin / canopy)                 | Model.kg                                                 | KX015-4.1470 / 1420               |\n|---:|:---------------------------------------------------------|:---------------------------------------------------------|:---------------------------------------------------------|:---------------------------------------------------------|:---------------------------------------------------------|:----------------------------------|\n|  0 | Operating weight *2 (cabin / canopy)                     | Operating weight *2 (cabin / canopy)                     | Operating weight *2 (cabin / canopy)                     | Operating weight *2 (cabin / canopy)                     | kg                                                       | 1545 / 1495 

In [27]:
Markdown(tables["6"][0])

|    | Model.Machine weight *1 (cabin / canopy)                 | Model.Machine weight *1 (cabin / canopy)                 | Model.Machine weight *1 (cabin / canopy)                 | Model.Machine weight *1 (cabin / canopy)                 | Model.kg                                                 | KX015-4.1470 / 1420               |
|---:|:---------------------------------------------------------|:---------------------------------------------------------|:---------------------------------------------------------|:---------------------------------------------------------|:---------------------------------------------------------|:----------------------------------|
|  0 | Operating weight *2 (cabin / canopy)                     | Operating weight *2 (cabin / canopy)                     | Operating weight *2 (cabin / canopy)                     | Operating weight *2 (cabin / canopy)                     | kg                                                       | 1545 / 1495                       |
|  1 |                                                          | Model                                                    | Model                                                    | Model                                                    | Model                                                    | D782-E3-BH                        |
|  2 |                                                          | Type                                                     | Type                                                     | Type                                                     | Type                                                     | Water-cooled,diesel engine,E-TVCS |
|  3 |                                                          | Output ISO14396                                          | Output ISO14396                                          | Output ISO14396                                          | PS (kW)/rpm                                              | 13.3 (9.8) / 2300                 |
|  4 | Engine                                                   | Output ISO9249 NET                                       | Output ISO9249 NET                                       | Output ISO9249 NET                                       | PS (kW)/rpm                                              | 13.1 (9.6) / 2300                 |
|  5 |                                                          | Number of cylinders                                      | Number of cylinders                                      | Number of cylinders                                      | Number of cylinders                                      | 3                                 |
|  6 |                                                          | Bore × Stroke                                            | Bore × Stroke                                            | Bore × Stroke                                            | mm                                                       | 67 × 73.6                         |
|  7 |                                                          | Displacement                                             | Displacement                                             | Displacement                                             | cc                                                       | 778                               |
|  8 |                                                          | Overall width                                            | Overall width                                            | Overall width                                            | mm                                                       | 990                               |
|  9 |                                                          | Overall height (cabin / canopy)                          | Overall height (cabin / canopy)                          | Overall height (cabin / canopy)                          | mm                                                       | 2350 / 2330                       |
| 10 |                                                          | Overall length                                           | Overall length                                           | Overall length                                           | mm                                                       | 3710                              |
| 11 |                                                          | Ground clearance                                         | Ground clearance                                         | Ground clearance                                         | mm                                                       | 160                               |
| 12 | Dimensions                                               | Dozer size (width × height)                              | Dozer size (width × height)                              | Dozer size (width × height)                              | mm                                                       | 990 × 230                         |
| 13 |                                                          | Rubber shoe width                                        | Rubber shoe width                                        | Rubber shoe width                                        | mm                                                       | 230                               |
| 14 |                                                          | Minimum front swivel radius                              | Minimum front swivel radius                              | Minimum front swivel radius                              | mm                                                       | 1490                              |
| 15 |                                                          | Boom swing angle (left / right) deg                      | Boom swing angle (left / right) deg                      | Boom swing angle (left / right) deg                      | Boom swing angle (left / right) deg                      | 75 / 60                           |
| 16 | Hydraulic                                                | P1, P2                                                   |                                                          |                                                          |                                                          | Variable displacement pump        |
| 17 | Hydraulic                                                |                                                          | Flow rate /min                                           | Flow rate /min                                           | Flow rate /min                                           | 16.6 × 2                          |
| 18 | Hydraulic                                                |                                                          | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | 20.6 (210)                        |
| 19 | Hydraulic                                                | P3                                                       |                                                          |                                                          |                                                          | Gear pump                         |
| 20 | Hydraulic                                                |                                                          | Flow rate /min                                           | Flow rate /min                                           | Flow rate /min                                           | 10.4                              |
| 21 | System                                                   |                                                          | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | 20.1 (205)                        |
| 22 | Hydraulic                                                | Auxiliary                                                | /min Max. flow rate                                      | /min Max. flow rate                                      | /min Max. flow rate                                      | 27.0                              |
| 23 | Hydraulic                                                | (AUX)                                                    | MPa (kgf/cm 2 ) Max. hydr. pressure                      | MPa (kgf/cm 2 ) Max. hydr. pressure                      | MPa (kgf/cm 2 ) Max. hydr. pressure                      | 20.6 (210)                        |
| 24 | Hydraulic                                                | Max.                                                     | digging                                                  | arm                                                      | kN (kgf)                                                 | 7.3 (740)                         |
| 25 | Hydraulic                                                | force                                                    |                                                          | bucket                                                   | kN (kgf)                                                 | 12.7 (1300)                       |
| 26 | Hydraulic                                                | Hydraulic reservoir (full)                               | Hydraulic reservoir (full)                               | Hydraulic reservoir (full)                               | Hydraulic reservoir (full)                               | 28                                |
| 27 | Max. travelling speed km/h                               | Max. travelling speed km/h                               | Max. travelling speed km/h                               | Max. travelling speed km/h                               | Max. travelling speed km/h                               | 2.1                               |
| 28 | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | 25.5 (0.26) / 24.5 (0.25)         |
| 29 | Swivelling speed rpm                                     | Swivelling speed rpm                                     | Swivelling speed rpm                                     | Swivelling speed rpm                                     | Swivelling speed rpm                                     | 9.1                               |
| 30 | Fuel tank capacity                                       | Fuel tank capacity                                       | Fuel tank capacity                                       | Fuel tank capacity                                       | Fuel tank capacity                                       | 21                                |
| 31 | LpA dB (A)                                               | LpA dB (A)                                               | LpA dB (A)                                               | LpA dB (A)                                               | LpA dB (A)                                               | 78                                |
| 32 | Noise level                                              | LwA                                                      |                                                          | (2000/14/EC)                                             | dB (A)                                                   | 93                                |
| 33 |                                                          | Hand arm (ISO5349-2:2001 )                               | system                                                   | Digging                                                  | m/s 2 RMS                                                | <2.5                              |
| 34 |                                                          |                                                          |                                                          | Levelling                                                | m/s 2 RMS                                                | <2.5                              |
| 35 |                                                          |                                                          |                                                          | Driving                                                  | m/s 2 RMS                                                | <2.5                              |
| 36 | Vibration* 3 KX015-4                                     |                                                          |                                                          | Idling                                                   | m/s 2 RMS                                                | <2.5                              |
| 37 |                                                          | Whole body (ISO2631- 1:1997) canopy                      |                                                          | Digging Lifting                                          | m/s 2 RMS m/s 2 RMS point radius                         | <0.5 <0.5 (2m)                    |
| 38 | Lift Point                                               | Height                                                   | Levelling                                                | Driving Over-front                                       | m/s 2 RMS                                                | <0.5                              |
| 39 |                                                          |                                                          | Blade                                                    | Idling Down                                              | m/s 2 RMS Blade UP                                       | <0.5 Over-side Blade              |

In [28]:
Markdown(tables["6"][1])

|    | KX015-4 cabin Cabin, Rubber version..Lift Point Height.   | Lifting point radius (2m).Over-front.Blade Down   | Lifting point radius (2m).Over-front.Blade UP   | Lifting point radius (2m).Over-side.   | Lifting point radius (max.).Over-front.Blade Down   | Lifting point radius (max.).Over-front.Blade UP   | kN (ton).Lifting point radius (max.).Over-side.Lift Point   |
|---:|:----------------------------------------------------------|:--------------------------------------------------|:------------------------------------------------|:---------------------------------------|:----------------------------------------------------|:--------------------------------------------------|:------------------------------------------------------------|
|  0 | 1.5m                                                      | 3.0 (0.30)                                        | 3.4 (0.35)                                      | 2.5 (0.26)                             | -                                                   | -                                                 | -                                                           |
|  1 | 1.0m                                                      | 4.5 (0.46)                                        | 3.5 (0.36)                                      | 2.3 (0.24)                             | -                                                   | -                                                 | -                                                           |
|  2 | 0.5m                                                      | 5.4 (0.55)                                        | 3.3 (0.34)                                      | 2.1 (0.22)                             | 2.6 (0.27)                                          | 1.6 (0.17)                                        | 1.1 (0.11) Lift Point Height                                |
|  3 | 0m                                                        | 5.3 (0.54)                                        | 3.2 (0.32)                                      | 2.0 (0.21)                             | -                                                   | -                                                 | -                                                           |

# 테이블 셀병합 부분 개선 검토

In [31]:
import os
import pandas as pd
from docling.document_converter import DocumentConverter
from time import time
from tqdm.auto import tqdm
from collections import defaultdict

file_path = filepath
lv1_cat, lv2_cat = "CE", "KUBOTA"

path = file_path.replace("\\", "/")
filename = path.split("/")[-1]

# -------------------------------
# span-aware table normalization
# -------------------------------
def table_to_dataframe_with_span(table):
    rows = table.rows

    # 최대 row / col 계산
    max_row = 0
    max_col = 0

    for r_idx, row in enumerate(rows):
        c_idx = 0
        for cell in row.cells:
            rs = cell.row_span or 1
            cs = cell.col_span or 1
            max_row = max(max_row, r_idx + rs)
            max_col = max(max_col, c_idx + cs)
            c_idx += cs

    grid = [[None for _ in range(max_col)] for _ in range(max_row)]

    # grid 채우기
    for r_idx, row in enumerate(rows):
        c_idx = 0
        for cell in row.cells:
            text = cell.text.strip() if cell.text else ""
            rs = cell.row_span or 1
            cs = cell.col_span or 1

            for dr in range(rs):
                for dc in range(cs):
                    rr = r_idx + dr
                    cc = c_idx + dc
                    if grid[rr][cc] is None:
                        grid[rr][cc] = text

            c_idx += cs

    df = pd.DataFrame(grid)

    # ---- 좌측 컬럼 forward-fill
    df = df.replace("", pd.NA)
    df.iloc[:, 0] = df.iloc[:, 0].ffill()

    return df



# -------------------------------
# main
# -------------------------------
start_time = time()

doc_converter = DocumentConverter()
conv_res = doc_converter.convert(file_path)

tables = defaultdict(list)

for table_ix, table in tqdm(enumerate(conv_res.document.tables)):
    page_num = table.prov[0].page_no if table.prov else "Unknown"

    try:
        df = table_to_dataframe_with_span(table)
        extracted_table = df.to_markdown(index=False)
    except Exception as e:
        extracted_table = f"⚠️ Table parse failed: {e}"

    tables[str(page_num)].append(extracted_table)

end_time = time() - start_time
print(f"Document converted and tables exported in {end_time:.2f} seconds.")


2026-01-18 13:51:11,964 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]


2026-01-18 13:51:12,015 - INFO - Going to convert document batch...
2026-01-18 13:51:12,016 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2026-01-18 13:51:12,019 - INFO - rapidocr cannot be used because onnxruntime is not installed.
2026-01-18 13:51:12,020 - INFO - Accelerator device: 'cpu'
2026-01-18 13:51:14,210 - INFO - Auto OCR model selected easyocr.
2026-01-18 13:51:14,212 - INFO - Accelerator device: 'cpu'
2026-01-18 13:51:15,646 - INFO - Accelerator device: 'cpu'
2026-01-18 13:51:16,111 - INFO - Processing document kubota_sample.pdf
d:\auto_vectordb\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
d:\auto_vectordb\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pi

Document converted and tables exported in 161.67 seconds.


In [32]:
tables = dict(tables)
tables

{'6': ["⚠️ Table parse failed: 'TableItem' object has no attribute 'rows'",
  "⚠️ Table parse failed: 'TableItem' object has no attribute 'rows'"]}